# Healome Aging Clock — Demo

Predict your biological age from standard blood biomarkers in under 2 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikhilYadala/healome_bio_age/blob/main/notebooks/demo.ipynb)

In [ ]:
# Install (uncomment for Colab — clone repo first, then run from repo root)
# !git clone https://github.com/nikhilYadala/healome_bio_age.git
# !cd healome_bio_age && pip install -e .
# Or if already in repo: !pip install -e .

import sys
sys.path.insert(0, '..')

from healome_clock import HealomeClock, predict_age, STANDARD_21_FEATURES, FEATURE_NAMES

## Quick Start — Predict from a blood panel

In [ ]:
# Example blood panel (values from a real NHANES participant)
blood_panel = {
    "LBXMCVSI": 99.0,    # Mean cell volume
    "LBXGH": 5.4,        # Glycohemoglobin (HbA1c)
    "LBXSATSI": 15.0,    # ALT
    "LBXRBCSI": 3.86,    # Red blood cell count
    "MCQ220": 2,          # Cancer history (2=No)
    "LBXPLTSI": 226.0,   # Platelet count
    "LBXSLDSI": 101.0,   # LDH
    "MCQ160D": 2,         # Angina history (2=No)
    "LBXLYPCT": 30.0,    # Lymphocyte %
    "LBDLYMNO": 2.0,     # Lymphocyte count
    "LBXSCK": 100.0,     # CPK
    "LBXSCR": 0.80,      # Creatinine
    "MCQ160A": 2,         # Arthritis history (2=No)
    "LBXSAPSI": 41.0,    # ALP
    "MCQ500": 2,          # Liver condition (2=No)
    "LBXSKSI": 4.0,      # Potassium
    "LBXRDW": 14.5,      # RDW
    "LBXMOPCT": 5.7,     # Monocyte %
    "LBXSBU": 7.0,       # BUN
    "MCQ550": 2,          # Gallstones (2=No)
    "LBXSGL": 79.0,      # Glucose
}

result = predict_age(blood_panel, chronological_age=45)
print(result.summary())

## Understanding the features

The 21-feature standard model uses **16 lab biomarkers** from routine blood work
plus **5 medical history questions** from the NHANES MCQ questionnaire.

In [ ]:
print("21-Feature Standard Model:\n")
print("Lab Biomarkers:")
for f in STANDARD_21_FEATURES:
    if not f.startswith("MCQ"):
        print(f"  {f:12s}  {FEATURE_NAMES[f]}")

print("\nQuestionnaire Items:")
for f in STANDARD_21_FEATURES:
    if f.startswith("MCQ"):
        print(f"  {f:12s}  {FEATURE_NAMES[f]}")

## Feature Importance

Which biomarkers matter most for predicting biological age?

In [ ]:
import matplotlib.pyplot as plt

clock = HealomeClock(variant="standard")
importances = clock.feature_importances()

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = range(len(importances))
ax.barh(y_pos, importances["importance"].values[::-1], color="#2563EB", alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(importances["name"].values[::-1], fontsize=10)
ax.set_xlabel("Feature Importance")
ax.set_title("What drives biological age prediction?", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Predict from a CSV file

In [ ]:
import pandas as pd

sample = pd.read_csv("../data/sample_input.csv")
print(f"Sample data: {sample.shape[0]} records")

results = clock.predict(sample)
if isinstance(results, list):
    for i, r in enumerate(results):
        print(f"  Record {i+1}: biological age = {r.biological_age:.1f}")
else:
    print(f"  Biological age: {results.biological_age:.1f}")

## Extended Model (35 features)

If you have a more comprehensive blood panel, the extended model uses 35 features.

In [ ]:
clock_ext = HealomeClock(variant="extended")
print(clock_ext)
print(f"\nAdditional features beyond the 21-feature model:")
from healome_clock import EXTENDED_35_FEATURES
extra = set(EXTENDED_35_FEATURES) - set(STANDARD_21_FEATURES)
for f in sorted(extra):
    print(f"  {f:12s}  {FEATURE_NAMES.get(f, '')}")

---

**Next steps:**
- See `training.ipynb` for the full training pipeline
- See `evaluation.ipynb` for benchmarking against other models
- Read [MODEL_CARD.md](../MODEL_CARD.md) for full model documentation
- Read [LIMITATIONS.md](../LIMITATIONS.md) before interpreting results

*Not a medical device. Not intended for clinical decision-making without professional oversight.*